In [5]:
import pandas as pd
import requests
import gzip
from io import BytesIO

# =============================
# PARAMÈTRES GÉNÉRAUX
# =============================
START_YEAR = 1900
END_YEAR = 2025
OUTPUT_CSV = "france_monthly_mean_temperature_1900_2025.csv"

# =============================
# SOURCES MÉTÉO-FRANCE
# =============================
# Archive historique (OK jusqu'à 1949)
URL_HISTORIQUE = (
    "https://www.data.gouv.fr/fr/datasets/r/"
    "f0c91372-01a5-4cb3-b200-ecf1b60c4aab"
)

# Données mensuelles récentes (1950 → présent)
# 👉 À récupérer UNE FOIS sur data.gouv.fr (voir explications plus bas)
URL_RECENTE = (
    "https://www.data.gouv.fr/fr/datasets/r/351b02dc-35d7-4fe9-ad0c-78d7443e8fcc"
)

URLS = [URL_HISTORIQUE, URL_RECENTE]

# =============================
# FONCTION DE CHARGEMENT
# =============================
def load_monthly_station_data(url: str) -> pd.DataFrame:
    print(f"Téléchargement : {url}")
    r = requests.get(url)
    r.raise_for_status()

    with gzip.open(BytesIO(r.content), "rt", encoding="utf-8") as f:
        df = pd.read_csv(f, sep=";", low_memory=False)

    df.columns = df.columns.str.lower()
    return df


# =============================
# CHARGEMENT DES DONNÉES
# =============================
dfs = []
for url in URLS:
    if "REMPLACER_PAR_RESOURCE_ID" not in url:
        dfs.append(load_monthly_station_data(url))

df = pd.concat(dfs, ignore_index=True)

# =============================
# PRÉPARATION DES DATES
# =============================
df["year"] = df["aaaamm"].astype(str).str[:4].astype(int)
df["month"] = df["aaaamm"].astype(str).str[4:6].astype(int)

df = df[
    (df["year"] >= START_YEAR) &
    (df["year"] <= END_YEAR) &
    (df["tm"].notna())
]

# =============================
# AGRÉGATION NATIONALE (TYPE 1)
# =============================
monthly_mean = (
    df
    .groupby(["year", "month"])["tm"]
    .mean()
    .reset_index()
)

pivot = monthly_mean.pivot(
    index="year",
    columns="month",
    values="tm"
)

pivot.columns = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

pivot = pivot.reset_index().sort_values("year")

# =============================
# EXPORT CSV
# =============================
pivot.to_csv(
    OUTPUT_CSV,
    index=False,
    float_format="%.2f"
)

print("===================================")
print("CSV généré :", OUTPUT_CSV)
print("Période :", pivot["year"].min(), "-", pivot["year"].max())
print("===================================")


Téléchargement : https://www.data.gouv.fr/fr/datasets/r/f0c91372-01a5-4cb3-b200-ecf1b60c4aab
Téléchargement : https://www.data.gouv.fr/fr/datasets/r/351b02dc-35d7-4fe9-ad0c-78d7443e8fcc
CSV généré : france_monthly_mean_temperature_1900_2025.csv
Période : 1900 - 1949
